# 🎭 Face2FaceRHO — Google Colab 실행 가이드

**Face2Face^ρ (ECCV2022)** — 실시간 고해상도 One-shot 얼굴 리인액트먼트

> ⚠️ **런타임 설정 확인**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택

---
### 📋 전체 파이프라인
1. 환경 설정 (레포 클론 + 패키지 설치)
2. 외부 모델 다운로드 (FLAME, DECA, Pre-trained weights)
3. DECA로 3DMM 계수 추출
4. 얼굴 리인액트먼트 실행

## ① 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

Fri Mar 20 04:16:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 레포 클론
!git clone https://github.com/NetEase-GameAI/Face2FaceRHO.git
%cd Face2FaceRHO

Cloning into 'Face2FaceRHO'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 119 (delta 8), reused 116 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 6.61 MiB | 12.08 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Face2FaceRHO


In [ ]:
# 필수 패키지 설치
!pip install torch==1.9.0+cu111 torchvision==0.10.0+cu111 -f https://download.pytorch.org/whl/torch_stable.html
!pip install face-alignment==1.3.4
!pip install kornia==0.5.8
!pip install yacs
!pip install ninja
!pip install cython
!pip install scikit-image
!pip install opencv-python
!pip install trimesh
!pip install PyYAML

# DECA 의존성
!pip install pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py38_cu111_pyt190/download.html

Looking in links: https://download.pytorch.org/whl/torch_stable.html
ERROR: Could not find a version that satisfies the requirement torch==1.9.0+cu111 (from versions: 2.2.0, 2.2.0+cpu, 2.2.0+cpu.cxx11.abi, 2.2.0+cu118, 2.2.0+cu121, 2.2.0+rocm5.6, 2.2.0+rocm5.7, 2.2.1, 2.2.1+cpu, 2.2.1+cpu.cxx11.abi, 2.2.1+cu118, 2.2.1+cu121, 2.2.1+rocm5.6, 2.2.1+rocm5.7, 2.2.2, 2.2.2+cpu, 2.2.2+cpu.cxx11.abi, 2.2.2+cu118, 2.2.2+cu121, 2.2.2+rocm5.6, 2.2.2+rocm5.7, 2.3.0, 2.3.0+cpu, 2.3.0+cpu.cxx11.abi, 2.3.0+cu118, 2.3.0+cu121, 2.3.0+rocm5.7, 2.3.0+rocm6.0, 2.3.1, 2.3.1+cpu, 2.3.1+cpu.cxx11.abi, 2.3.1+cu118, 2.3.1+cu121, 2.3.1+rocm5.7, 2.3.1+rocm6.0, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==1.9.0+cu111
  Preparing metadata (setup.py) ... done
  Created wheel for face-alignment: filename=face_alignment-1.3.4-py2.py3-none-any.whl size=27848 sha256=2c19a87f4072f728cf70854b7581b48cb827d07bb92edf1250f09d3a11e32ad2
  Stored

In [ ]:
# pytorch3d 설치가 실패할 경우 소스에서 빌드 (시간이 오래 걸림 ~10분)
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

try:
    import pytorch3d
    print("✅ pytorch3d 설치 완료:", pytorch3d.__version__)
except ImportError:
    print("⚠️ pytorch3d 미설치 → 소스 빌드 시작 (약 10분 소요)")
    !pip install "git+https://github.com/facebookresearch/pytorch3d.git@v0.6.0"

## ② 외부 모델 다운로드

> ### ⚠️ 수동 다운로드 필요 (아래 3가지)
>
> | 모델 | 링크 | 저장 위치 |
> |------|------|----------|
> | **Pre-trained weights** | [Google Drive](https://drive.google.com/drive/folders/1eKHMevJBIvjLcVBBB2EkJwHkzN7pTLJG) | `./src/checkpoints/voxceleb_face2facerho/` |
> | **FLAME 2020** | [flame.is.tue.mpg.de](https://flame.is.tue.mpg.de/) → Downloads → FLAME 2020 | `./src/external/data/generic_model.pkl` |
> | **DECA model** | [DECA GitHub](https://github.com/yfeng95/DECA) → Downloads | `./src/external/data/deca_model.tar` |
>
> 다운로드 후 아래 셀에서 Google Drive를 마운트하거나 직접 업로드하세요.

In [ ]:
# Google Drive 마운트 (모델 파일을 Drive에 저장해 둔 경우)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# 디렉토리 생성
os.makedirs('./src/checkpoints/voxceleb_face2facerho', exist_ok=True)
os.makedirs('./src/external/data', exist_ok=True)

# ======================================================
# 아래 경로를 본인의 Google Drive 경로에 맞게 수정하세요
# ======================================================

DRIVE_BASE = '/content/drive/MyDrive/Face2FaceRHO_models'  # ← 수정

# Pre-trained weights 복사
!cp -r "{DRIVE_BASE}/voxceleb_face2facerho/" ./src/checkpoints/

# FLAME model 복사
!cp "{DRIVE_BASE}/generic_model.pkl" ./src/external/data/

# DECA model 복사
!cp "{DRIVE_BASE}/deca_model.tar" ./src/external/data/

print("✅ 모델 파일 복사 완료")

In [ ]:
# 파일 존재 여부 확인
import os

checks = [
    './src/checkpoints/voxceleb_face2facerho',
    './src/external/data/generic_model.pkl',
    './src/external/data/deca_model.tar',
]

for path in checks:
    exists = os.path.exists(path)
    icon = '✅' if exists else '❌'
    print(f"{icon} {path}")

## ③ 테스트 케이스로 바로 실행 (사전 제공 데이터 사용)

In [ ]:
# 테스트 케이스 파일 확인
!find ./test_case -type f | sort

In [ ]:
# 출력 디렉토리 생성
!mkdir -p ./test_case/result

# =======================================
# 방법 A: 사전 제공된 headpose/landmark 사용 (빠른 테스트)
# =======================================
!python src/reenact.py \
    --config ./src/config/test_face2facerho.ini \
    --src_img ./test_case/source/source.jpg \
    --src_headpose ./test_case/source/original/headpose.txt \
    --src_landmark ./test_case/source/original/landmark.txt \
    --drv_headpose ./test_case/driving/original/headpose.txt \
    --drv_landmark ./test_case/driving/original/landmark.txt \
    --output_dir ./test_case/result

print("\n✅ 리인액트먼트 완료!")

In [ ]:
# 결과 이미지 확인
from IPython.display import Image, display
import os

result_path = './test_case/result/result.png'
if os.path.exists(result_path):
    print("🎉 결과 이미지:")
    display(Image(result_path, width=512))
else:
    print("❌ result.png를 찾을 수 없습니다. 에러 로그를 확인하세요.")

## ④ DECA를 이용한 3DMM 계수 추출 (커스텀 이미지 사용 시)

In [ ]:
# DECA 서브모듈 초기화
!git submodule update --init --recursive

In [ ]:
# 커스텀 이미지 업로드
from google.colab import files
import shutil, os

print("소스 이미지(source face)를 업로드하세요:")
uploaded = files.upload()

os.makedirs('./custom_input/source', exist_ok=True)
for fname in uploaded:
    shutil.copy(fname, f'./custom_input/source/{fname}')
    print(f"✅ 저장됨: ./custom_input/source/{fname}")

In [ ]:
# 드라이빙 이미지 업로드
print("드라이빙 이미지(driving face)를 업로드하세요:")
uploaded_drv = files.upload()

os.makedirs('./custom_input/driving', exist_ok=True)
for fname in uploaded_drv:
    shutil.copy(fname, f'./custom_input/driving/{fname}')
    print(f"✅ 저장됨: ./custom_input/driving/{fname}")

In [ ]:
# DECA로 3DMM 계수 추출
SRC_IMG = './custom_input/source/source.jpg'  # ← 파일명 수정
DRV_IMG = './custom_input/driving/driving.jpg'  # ← 파일명 수정

!python src/fit_3dmm.py \
    --src_img {SRC_IMG} \
    --drv_img {DRV_IMG} \
    --output_dir ./custom_input/coeffs

print("\n✅ 3DMM 계수 추출 완료")

In [ ]:
# 커스텀 이미지로 리인액트먼트 실행
!mkdir -p ./custom_input/result

SRC_IMG = './custom_input/source/source.jpg'  # ← 수정

!python src/reenact.py \
    --config ./src/config/test_face2facerho.ini \
    --src_img {SRC_IMG} \
    --src_headpose ./custom_input/coeffs/source/headpose.txt \
    --src_landmark ./custom_input/coeffs/source/landmark.txt \
    --drv_headpose ./custom_input/coeffs/driving/headpose.txt \
    --drv_landmark ./custom_input/coeffs/driving/landmark.txt \
    --output_dir ./custom_input/result

print("\n✅ 완료!")

# 결과 표시
from IPython.display import Image, display
display(Image('./custom_input/result/result.png', width=512))

## ⑤ 결과 다운로드

In [ ]:
from google.colab import files

# 테스트 케이스 결과 다운로드
files.download('./test_case/result/result.png')

# 커스텀 결과 다운로드 (사용한 경우)
# files.download('./custom_input/result/result.png')

---
## 🛠️ 트러블슈팅

| 에러 | 해결 방법 |
|------|----------|
| `ModuleNotFoundError: pytorch3d` | 셀 ①의 소스 빌드 셀 실행 |
| `FileNotFoundError: generic_model.pkl` | FLAME 모델 경로 재확인 |
| `CUDA out of memory` | 런타임 재시작 후 GPU 메모리 확보 |
| `No GPU found` | 런타임 → 런타임 유형 변경 → GPU 선택 |
| `deca_model.tar not found` | DECA GitHub에서 직접 다운로드 필요 |

### 📌 참고 링크
- [Face2FaceRHO GitHub](https://github.com/NetEase-GameAI/Face2FaceRHO)
- [FLAME 모델 다운로드](https://flame.is.tue.mpg.de/)
- [DECA GitHub](https://github.com/yfeng95/DECA)